# Part 1: Ready Made vs Custom Made Data

## 1.1 Pros and cons of the structure
Centola’s custom-made experiment gives researchers more control. They design the network structure, exposure, and how outcomes are measured. That makes it much easier to draw causal conclusions, because fewer outside factors interfere. The data are also cleaner and more directly linked to the theory being tested. 
The downside is that the setting is artificial. Participants know they are in a study, and the task usually takes place in a simplified online environment. Because of that, the results might not fully reflect how people behave in messy, real-world situations. The sample size is also smaller and may not represent the broader population. 
Where Nicolaides uses data that were not created for research, but come from real-world digital activity. This means the data are large-scale and reflect actual behavior over time. However, these data can be incomplete, biased, or shaped by platform algorithms, and researchers have less control over how they were produced.

## 1.2 Influence on development and experiments
Since Centola’s experiment has a high level of control, that means we can be more confident about causality. When the outcome changed after changing only one variable we can reasonably say the cause. However, since the setting is simplified and experimental, we should be cautious about assuming the same effects will appear in everyday life. 
On the other hand, in Nicolaides' study the findings reflect real-world behavior at scale, which increases the external relevance. Although since the data wasn't designed for research, the observed patterns may be influenced by hidden biases. Therefore, the results are more correlational and require more careful interpretation. 



# Part 2: Find Researchers using the OpenAlex API

## 2.1 Data Retrieval

Using the list of researcher names from Week 1, we query the OpenAlex `authors` endpoint.  
For each author, we retrieve:

- `id`
- `display_name`
- `works_api_url`
- `h_index`
- `works_count`
- `country_code`

In [24]:
import pandas as pd # Dataframe library
from dotenv import load_dotenv # Loading the API key from a .env file
import requests # used to make API queries
import os
from datetime import datetime # Current date and time for logs and saving timestamped files
import backoff # https://pypi.org/project/backoff/
from tqdm.contrib.concurrent import thread_map # Used for parallel processing
import json # Json used to import scraped names from week 1
from collections import Counter # For taking the mode of a list
import Levenshtein as L # Distance metric
from rapidfuzz import fuzz # For tokenizing words
from itertools import chain 
from time import sleep # sleep for some ms in the query to avoid limit of 100 requests per second

In [25]:
# ==========================
#   Load the scraped names
# ==========================
with open("scraped_names.json", "r", encoding="utf-8") as f:
    scraped_names = json.load(f)

scraped_names = scraped_names
print("How many researcher names are scraped (from week 1)?:", len(scraped_names))

How many researcher names are scraped (from week 1)?: 1572


In [26]:
# =====================================
#  Helper Functions for the API Queries
# =====================================

# -- Method for logging to a .txt file (no practical purpose other than debugging!) -- 
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
LOG_PATH = f"./logs/log_{timestamp}.txt"

def log_line(identifier, logText):
    """
    identifier: something that identifies the logText preferably uniquely for debugging.
    """
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{identifier}\t{logText}\n")
        f.flush()


# -- API backoff --
# -- To avoid the limit of 100 requests per second and other basic errors when querying the API
# -- Backoff will make slight delays in case an error is returned and then try again after a short wait for max 7 times etc., to make the code more robust.
# Retry on all RequestExceptions, exponential backoff with jitter.
@backoff.on_exception(
    backoff.expo,
    requests.exceptions.RequestException,
    max_tries=7,                     # default infinite — better to cap
    max_time=60,                     # stop after 60 sec
)
def get_with_backoff(url, **kwargs):
    resp = requests.get(url, **kwargs)

    if resp.status_code in (429, 500, 502, 503, 504):
        log_line(resp.status_code, resp.text)
        resp.raise_for_status()

    return resp


# -- Method for retrieving the desired author-information from the API query output -- 
def getPersonInfo(p : dict):
    oid  = (p.get("id") or None)
    sid  = oid[-11:] if len(oid) >= 11 else oid

    display_name = p.get("display_name")
    
    works_api_url = p.get("works_api_url")
    
    h_index = (p.get("summary_stats") or {}).get("h_index")
    
    works_count = p.get("works_count")
    
    # Safe mode of last_known_institutions.country_code
    insts = p.get("last_known_institutions") or []
    countries = [i.get("country_code") for i in insts if (i or {}).get("country_code")]
    country_code = Counter(countries).most_common(1)[0][0] if countries else None

    cited_by_count = p.get("cited_by_count")

    return sid, display_name, works_api_url, h_index, works_count, country_code, cited_by_count

# -- Get and save the person with the maximum number of works and citations --
def getMaxPerson(p : dict, maxworks : int, maxcites : int, maxPerson):

    if p["works_count"] > maxworks: # If works_count is greater than the current for the scraped name
        maxPerson = getPersonInfo(p)
        maxworks = p["works_count"]

    elif p["works_count"] == maxworks: # If works_count is equal to the current for the scraped name (tie break)
        if p["cited_by_count"] > maxPerson[-1]:
            maxPerson = getPersonInfo(p)

    return maxworks, maxcites, maxPerson

# -- Method for making name tokens for recognizing names like "John Smith" and "Smith John" --
def fuzzy_name_match(a, b):
    score = max(
        fuzz.token_sort_ratio(a, b),
        fuzz.token_set_ratio(a, b)
    )
    return score

# -- Method for removing commas, dots and apostrophes etc. in names like "John Smith" and "Smith, John" so they're properly tokenized --
def cleanName(s):
    s = s.lower()
    for ch in [",", ".", "'", '"']:
        s = s.replace(ch, "")
    return " ".join(s.split())

# --------------------------------------------
# -- Method handling the API query results ---
def handle_results(response_json, name):
    maxworks = 0
    maxcites = 0
    maxPers = None
    
    results = response_json.get("results") or []

    name_cf = cleanName(name)

    for p in results:
        alt_list = p.get("display_name_alternatives") or []
        alts_cf = [cleanName(alt) for alt in alt_list]
        display_cf = cleanName(p["display_name"])

        # Check if there's an exact match in display_name or display_name_alternatives
        is_exact = display_cf == name_cf or name_cf in alts_cf
        
        if is_exact:
            maxPers = getPersonInfo(p)
            break

        # IF there's no exact match: Check if there's a match when tokenizing the names of an author
        matched_fuzzy = (
            fuzzy_name_match(name_cf, display_cf) or
            any(fuzzy_name_match(name_cf, alt_cf) for alt_cf in alts_cf)
        )

        if matched_fuzzy > 90: # 90& threshold for the tokens match
            maxworks, maxcites, maxPers = getMaxPerson(p, maxworks, maxcites, maxPers)
            continue

        # IF there's no match in the tokenized names (like "John Smith" = "Smith, John") try the Levenshtein-distance match.
        dist = L.distance(name_cf, display_cf)
        norm = dist / max(len(name_cf), len(display_cf))

        if norm <= 0.25: # Threshold of .25% of the name (its less in practice than it sounds)
            maxworks, maxcites, maxPers = getMaxPerson(p, maxworks, maxcites, maxPers)

    # Create final person record
    if maxPers is not None:
        return maxPers
    else: # If no person is found anywhere in the queries add their name as a NaN row for now
        return [None, name, None, None, None, None, None]


In [27]:
# ====================================
#          API query method 
# ====================================
# This method is the method that handles querying the API. It includes both a cheap and expensive fallback query.
# NOTICE!: The API_KEY should be placed in the .env file which is loaded automatically. This is to avoid accidental information leakage to GitHub etc of private API keys.
# Add it to a file called ".env" exactly like: OPENALEX_API_KEY=api_key

def getAuthor(authorNames):
    load_dotenv(override=True)
    API_KEY = os.getenv("OPENALEX_API_KEY")

    BASE = "https://api.openalex.org/authors"
    people = {}

    for name in authorNames:

        # ----------------------------
        # 1. CHEAP QUERY (list query) Cost per query: $0.0001
        # ----------------------------
        cheap_params = {
            "filter": f'display_name:"{name}"',
            # "select": "id,display_name,works_api_url,summary_stats,works_count,last_known_institutions,cited_by_count",
            "per-page": 100,
            "api_key": API_KEY
        }

        r = requests.get(BASE, params=cheap_params).json()

        # print(r)

        if r.get("results"):
            people[name] = handle_results(r, name)
            continue

        # ------------------------------------------
        # 2. EXPENSIVE FALLBACK QUERY (search query)   !! -- Cost per query: $0.001
        # ------------------------------------------
        log_line(name, f"Cheap search failed. Trying expensive full search for: {name}")

        expensive_params = {
            "search": name,   # FULL TEXT SEARCH (expensive)
            "select": "id,display_name,works_api_url,summary_stats,works_count,last_known_institutions,cited_by_count",
            "per-page": 200,
            "api_key": API_KEY
        }

        r2 = requests.get(BASE, params=expensive_params).json()

        if r2.get("results"):
            people[name] = handle_results(r2, name)
        else:
            people[name] = [None, name, None, None, None, None, None]

    return people

In [33]:
# =============================================
# Parallel processing of API queries using tqdm
# =============================================
# ! - Run this code after the above blocks to actually begin querying the API in parallel - ! (took me 7:33 min to run all 1572 scraped names)

chunks = [[n] for n in ["Kathleen M. Carley"]]
all_results = thread_map(getAuthor, chunks, max_workers=8, chunksize=1, desc="Fetching") # Used $0.3827 of API queries

merged = {}
for d in all_results:
    merged.update(d)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
df = pd.DataFrame(merged.values(), columns=['id', 'display_name', "works_api_url", "h_index", "works_count", "country_code", "cited_by_count"])
# df.to_pickle(f"./datasets/OpenAlexAPI_researchers_{timestamp}.pkl")
df

Fetching: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


,id,display_name,works_api_url,h_index,works_count,country_code,cited_by_count
0,A5085927300,Kathleen M. Carley,https://api.openalex.org/works?filter=author.i...,80,882,US,28104


In [28]:
# Load the dataset of the 1572 authors queried just above.
df_name = "OpenAlexAPI_researchers_2026-03-02_16-02-10_final"
df = pd.read_pickle(f"./datasets/{df_name}.pkl")

# pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')

# This is a list of the names that returned no authors from the query under the chosen limitations and assumptions.
nanids = (df[df["id"].isna()][0:])[["display_name"]]
print(len(nanids))
nanids

90


,display_name
10,Adam (Zhengzi) Zhou
20,Amirhossein Nakhaei
34,IC2S2 Program Chairs
41,Vincent Christoph Brockers
66,Justyna Janczy
...,...
1529,Xinrui Chloe Zhao
1538,Ethel Elikem Mensah
1541,Joaquín Ignacio Villagra Pacheco
1568,Kwabena Afriyie Owusu


In [31]:
matches = [name for name in scraped_names if "kathleen" in name.lower()]
matches

['Kathleen Carly', 'Kathleen M. Carley']

In [32]:
df[df["display_name"] == "Kathleen M. Carley"]

,id,display_name,works_api_url,h_index,works_count,country_code,cited_by_count
571,A5085927300,Kathleen M. Carley,https://api.openalex.org/works?filter=author.i...,80.0,881.0,NaN,28063.0


## 2.2 Handling Challenges and Reflection  

While collecting data from OpenAlex, we encountered several challenges.

First, understanding the API response structure required careful inspection. We reviewed example JSON outputs and documentation to correctly extract nested fields such as summary_stats, last_known_institutions, and authorships.

Second, name ambiguity posed difficulties. For example, searching “David Wright” returned multiple candidates. We selected the profile with the highest works_count, assuming it most likely represents an active researcher. In more nuanced cases like “Ágnes Horvát”, accents and alternative spellings produced several close matches. We therefore checked display_name_alternatives and compared works_count before selecting the most plausible profile.

Third, missing values such as absent institutions or summary statistics caused errors. We handled this using .get() with default None values.

Finally, API daily limits required meticulous thinking about the exact structure of the query as different query types, like list and search queries, have different costs.

---

### One specific problem and solution  

A major issue was name ambiguity. Some scraped names matched multiple OpenAlex profiles with slight spelling differences, prefixes, or accents. For common names, we selected the profile with the highest works_count as a proxy for an established researcher. In more complex cases, we checked whether the scraped name appeared in display_name_alternatives and compared works_count across candidates before selecting the most plausible match.

We chose this approach because it is systematic, reproducible, and requires no manual intervention. A potential limitation is that less prolific researchers with common names may occasionally be misassigned. However, this rule substantially reduced incorrect matches compared to selecting the first API result.

# Part 3: Collect Research Articles

## 3.1 Data Overview and Reflection

The code queries the OpenAlex works endpoint for IC2S2 authors using batched author filters and concept constraints. It retrieves the required data, constructs datasets D2 and D3, removes duplicates, and saves them to file.

In [20]:
# Apply the filters to the author table so 5 <= works count <= 5,000 and not NaN
D1_filtered = df[
    df['works_count'].notna() &
    (df['works_count'] >= 5) &
    (df['works_count'] <= 5000)
]
D1_authorIDs = [id for id in D1_filtered["id"]] # Extract IDs as list and remove NaN IDs.

print("Number of authors with 5 <= work_count <= 5000:", len(D1_authorIDs))

Number of authors with 5 <= work_count <= 5000: 1225


In [ ]:
# =============================================
# Parallel processing of API queries using tqdm
# =============================================
# This API query queries all works found by an author under the given limitations.

load_dotenv(override=True)
API_KEY = os.getenv("OPENALEX_API_KEY")

BASE = "https://api.openalex.org/works"

def getWorksByAuthorIDs(author_list):
    ids = "|".join(author_list)

    css_ids = "C144024400|C15744967|C162324750|C17744445" # SOC|PSY|ECON|POLSCI
    quant_ids = "C33923547|C121332964|C41008148" # MATH|PHYS|CS

    params = {
            "filter": f"author.id:{ids},cited_by_count:>10,authors_count:<10,concepts.id:{css_ids},concepts.id:{quant_ids}",
            "select": "id,publication_year,cited_by_count,authorships,title,abstract_inverted_index",
            "per_page": 200,
            "api_key": API_KEY,
            "cursor":"*"
        }

    results = []
    max_pages = 50 #safety cap
    pages = 0

    # Loop to query all the pages of works by an author
    while True:

        if pages >= max_pages:
            print(f"Stopped after reaching safety cap of {max_pages} pages.")
            log_line(pages, f"Stopped after reaching safety cap of {max_pages} pages.")
            break

        r = get_with_backoff(BASE, params=params, timeout=60).json()

        page_results = r.get("results")
        if not page_results:
            break  # done
        results.extend(page_results)

        next_cursor = r.get("meta", {}).get("next_cursor")

        if not next_cursor:
            break  # done

        params["cursor"] = next_cursor
        pages += 1
        sleep(0.2) # polite pause - adjust according to rate limits

    return results


# Parallel processing of API queries using tqdm
chunks = [D1_authorIDs[i:i+25] for i in range(0, len(D1_authorIDs), 25)] # Split authors ino chunks (lists) of 25 authors in each for parallel processing
all_results = thread_map(getWorksByAuthorIDs, chunks, max_workers=8, chunksize=1, desc="Fetching")


# ===========================================
# ---- Save results as Pandas Dataframes ----
# ===========================================
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

flat_results = list(chain.from_iterable(all_results))
df = pd.DataFrame(flat_results)

D2 = df[["id", "publication_year", "cited_by_count", "authorships"]].copy()
D3 = df[["id", "title", "abstract_inverted_index"]].copy()

# Clean IDs
D2["id"] = D2["id"].map(lambda x: x.split("/")[-1] if isinstance(x, str) else x)
D3["id"] = D3["id"].map(lambda x: x.split("/")[-1] if isinstance(x, str) else x)


# Extract author IDs
D2["authors"] = D2["authorships"].apply(
    lambda xs: [] if xs is None else [
        a["author"]["id"].split("/")[-1]
        for a in xs
        if a["author"]["id"] is not None
    ]
)

D2 = D2.drop(columns=["authorships"])

D2.to_pickle(f"./datasets/D2_{timestamp}.pkl")
D3.to_pickle(f"./datasets/D3_{timestamp}.pkl")

D2.head()

Fetching: 100%|██████████| 49/49 [00:41<00:00,  1.19it/s]


,id,publication_year,cited_by_count,authors
0,W2055957857,1999.0,3185,[A5064753440]
1,W2095655043,2013.0,3013,"[A5090665793, A5113226689]"
2,W2002781392,2006.0,2510,"[A5064753440, A5083730471]"
3,W2096974619,2014.0,1831,"[A5082742221, A5113226689, A5002798141, A50314..."
4,W3124726575,2021.0,1016,"[A5020533147, A5076651767, A5052536455, A50215..."


In [14]:
D3.head()

,id,title,abstract_inverted_index
0,W2055957857,Framing as a Theory of Media Effects,"{'Research': [0], 'on': [1], 'framing': [2, 25..."
1,W2095655043,Text as Data: The Promise and Pitfalls of Auto...,"{'Politics': [0], 'and': [1, 9, 67, 96, 99, 10..."
2,W2002781392,"Framing, Agenda Setting, and Priming: The Evol...","{'This': [0], 'special': [1], 'issue': [2], 'o..."
3,W2096974619,Structural Topic Models for Open‐Ended Survey ...,"{'Collection': [0], 'and': [1, 14, 37, 78, 97,..."
4,W3124726575,Shifting attention to accuracy can reduce misi...,None


### Dataset Overview and Reflection Questions:

**Dataset summary:**

In [8]:
print("The dataset D2 contains", len(D2["id"].drop_duplicates()), "unique works.")
unique_ids = set().union(*D2['authors']) # Unite the ID lists into a SET (set items are always unique).
print("Number of unique researchers co-authoring the papers in D2 (IC2S2 papers):", len(unique_ids))

The dataset D2 contains 12318 unique works.
Number of unique researchers co-authoring the papers in D2 (IC2S2 papers): 19489


**Efficiency in code - strategies implemented to make the code more efficient:**

Besides *filtering* the queries according to the specifications set in the assignment - only IC2S2 authors with work count between 5 and 5.000, works having *more than* 10 citations, and *fewer than* 10 authors, and including only topics relevant to Computational Social Science - to increase efficiency only the desired attributes are *selected* to be retrieved in the query: *id, publication_year, cited_by_count, author_ids, title, abstract_inverted_index*.

Furthermore, instead of retrieving the works of a single author ID per query, a *bulk request* of 25 authers are included in each query and *cursor paging*, with 200 works per page, is implemented to handle the handle the load of works returned.

Finally, a tqdm thread_map is implemented to allow *parallel processing* of the requests, substantially increasing efficiency.

**Filtering Criteria and Dataset Relevance:**

The filters were chosen to balance relevance, quality, and size. Limiting authors to 5–5,000 works reduced noisy matches at the low end and prevented extreme dominance at the high end. Requiring `cited_by_count > 10` favors papers with demonstrated uptake, though it may underrepresent newer publications. Restricting to fewer than 10 authors per work reduces large consortium papers and keeps a clearer link to individual IC2S2 authors.  

The concept intersection (Sociology/Psychology/Economics/Political Science AND Mathematics/Physics/Computer Science) targets computational social science by combining social-science and quantitative domains.  

Overall, these choices likely overrepresent established, highly cited, smaller-team research and underrepresent newer work, large-team computational projects, and papers classified under closely related concepts.